# <font color="#418FDE" size="6.5" uppercase>**Lasso Elastic Net**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit Lasso and Elastic Net models and interpret sparse coefficients. 
- Use cross-validation to select alpha and l1_ratio values. 
- Analyze convergence warnings, coefficient paths, and sparsity-based feature selection. 


## **1. Sparse Regression**

### **1.1. Lasso Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_01_01.jpg?v=1783959366" width="250">



>* Lasso removes weak predictors by zeroing coefficients
>* Sparse models improve simplicity and interpretability

>* Penalty strength controls coefficient shrinkage
>* Selected predictors aid prediction, not causation

>* Read signs, zeros, and shrunk magnitudes carefully
>* Correlated features may hide shared importance



In [ ]:
#@title Python Code - Lasso Regression

# Demonstrate Lasso without external machine learning libraries.
# Sparse coefficients can remove weak predictors.
# Standardization makes penalty effects easier to compare.

import numpy as np
import matplotlib.pyplot as plt

# Create a small reproducible housing style dataset.
rng = np.random.default_rng(42)
n_samples, n_features = 80, 8

# Simulate predictors with mixed practical meanings.
X = rng.normal(size=(n_samples, n_features))
true_coef = np.array([45, 0, -30, 0, 18, 0, 0, 8])

# Build prices from only four useful predictors.
y = X @ true_coef + rng.normal(0, 12, n_samples)
assert X.shape == (n_samples, n_features)

# Standardize features and center the target.
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_scaled = (X - X_mean) / X_std

y_centered = y - y.mean()
alpha = 8.0
max_iter = 3000

# Define soft thresholding for Lasso updates.
def soft_threshold(value, penalty):
    if value > penalty:
        return value - penalty

    if value < -penalty:
        return value + penalty

    return 0.0

# Fit Lasso using simple coordinate descent.
coef = np.zeros(n_features)
for iteration in range(max_iter):
    old_coef = coef.copy()

    for j in range(n_features):
        prediction = X_scaled @ coef
        residual = y_centered - prediction + X_scaled[:, j] * coef[j]

        rho = np.mean(X_scaled[:, j] * residual)
        coef[j] = soft_threshold(rho, alpha)

    if np.max(np.abs(coef - old_coef)) < 1e-6:
        break

# Summarize sparse coefficient selection.
feature_names = [
    "sqft", "bedrooms", "distance", "age",
    "school", "garage", "tax", "renovation"
]

selected = [name for name, value in zip(feature_names, coef) if abs(value) > 1e-8]
zero_count = int(np.sum(np.abs(coef) <= 1e-8))

print(f"Manual Lasso alpha: {alpha}")
print(f"Iterations used: {iteration + 1}")
print(f"Selected features: {selected}")
print(f"Zero coefficients: {zero_count} of {n_features}")
print("Coefficients show prediction associations, not proven causes.")

# Plot coefficients to visualize sparsity.
colors = ["tab:blue" if abs(v) > 1e-8 else "lightgray" for v in coef]
plt.figure(figsize=(8, 4))
plt.bar(feature_names, coef, color=colors)
plt.axhline(0, color="black", linewidth=1)
plt.title("Lasso Regression Creates Sparse Coefficients")
plt.ylabel("Standardized coefficient")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()



### **1.2. Elastic Net**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_01_02.jpg?v=1783959370" width="250">



>* Combines Lasso sparsity with ridge stability
>* Keeps related predictors, removes weak ones

>* Nonzero coefficients show useful predictors
>* Shrinkage reduces overfitting and removes noise

>* Selects stable groups of related predictors
>* Zero coefficients reflect chosen regularization



### **1.3. Sparsity selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_01_03.jpg?v=1783959364" width="250">



>* Selects important predictors, zeros out weak ones
>* Creates simpler, more interpretable regression models

>* Nonzero coefficients indicate selected useful predictors
>* Zero coefficients lack unique value here

>* Sparse models improve clarity and focus
>* Interpret coefficients cautiously within the workflow



In [ ]:
#@title Python Code - Sparsity selection

# Sparse regression can select useful predictors.
# This example implements Lasso without sklearn.
# Coefficients near zero become excluded features.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set deterministic randomness for reproducible results.
rng = np.random.default_rng(42)

# Create a small synthetic housing dataset.
n_samples, n_features = 80, 8
X_raw = rng.normal(size=(n_samples, n_features))

# Define only three truly useful predictors.
true_beta = np.array([4.0, 0.0, -3.0, 0.0, 2.0, 0.0, 0.0, 0.0])
y = X_raw @ true_beta + rng.normal(scale=0.8, size=n_samples)

# Standardize features so penalties are fair.
X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0)
X = (X_raw - X_mean) / X_std

# Center the target for a simple intercept.
y_mean = y.mean()
y_centered = y - y_mean

# Validate the design matrix dimensions.
assert X.shape == (n_samples, n_features)
assert y_centered.shape[0] == n_samples

# Soft thresholding creates exact zeros.
def soft_threshold(value, penalty):
    return np.sign(value) * max(abs(value) - penalty, 0.0)

# Fit Lasso using coordinate descent.
def fit_lasso_cd(X, y, alpha, max_iter=1000, tol=1e-6):
    n_rows, n_cols = X.shape
    beta = np.zeros(n_cols)
    col_norms = (X * X).sum(axis=0) / n_rows

    for iteration in range(max_iter):
        old_beta = beta.copy()
        for j in range(n_cols):
            residual = y - X @ beta + X[:, j] * beta[j]
            rho = (X[:, j] @ residual) / n_rows
            beta[j] = soft_threshold(rho, alpha) / col_norms[j]

        if np.max(np.abs(beta - old_beta)) < tol:
            break

    return beta, iteration + 1

# Try several alpha values to show sparsity.
alphas = [0.02, 0.10, 0.30, 0.70]
rows = []

# Store active feature counts for each penalty.
for alpha in alphas:
    beta, steps = fit_lasso_cd(X, y_centered, alpha)
    active = int(np.sum(np.abs(beta) > 1e-8))
    rows.append((alpha, active, steps, beta))

# Print a compact sparsity summary.
print("alpha | active features | iterations")
for alpha, active, steps, beta in rows:
    print(f"{alpha:4.2f}  | {active:14d} | {steps:10d}")

# Show selected coefficients for one alpha.
chosen_alpha, active, steps, chosen_beta = rows[2]
feature_names = [f"feature_{i}" for i in range(n_features)]
selected = [name for name, b in zip(feature_names, chosen_beta) if abs(b) > 1e-8]

# Print the selected sparse feature set.
print(f"Selected at alpha={chosen_alpha:.2f}: {selected}")
print("Zero coefficients mean excluded predictors here.")

# Plot coefficient paths across alpha values.
coef_matrix = np.vstack([row[3] for row in rows])
plt.figure(figsize=(7, 4))

# Draw one path per feature.
for j, name in enumerate(feature_names):
    plt.plot(alphas, coef_matrix[:, j], marker="o", label=name)

# Label the sparse coefficient path plot.
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("alpha penalty strength")
plt.ylabel("standardized coefficient")
plt.title("Lasso sparsity selection path")

# Keep the legend compact and readable.
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()



## **2. Cross Validated Models**

### **2.1. Lasso Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_02_01.jpg?v=1783959378" width="250">



>* Choose Lasso regularization using validation
>* Balance overfitting, underfitting, and feature selection

>* Test alpha values across validation folds
>* Choose accurate, simpler sparse models

>* Balance prediction accuracy with sparse coefficients
>* Validate selected features using domain context



### **2.2. Elastic Net Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_02_02.jpg?v=1783959380" width="250">



>* Balances accuracy with interpretable feature selection
>* Cross-validation chooses Lasso-Ridge regularization strength

>* Tune penalty strength and Lasso-Ridge balance
>* Select settings that best generalize

>* Refit final model and inspect coefficients
>* Validate feature stability with domain context



### **2.3. Multitask Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_02_03.jpg?v=1783959386" width="250">



>* Learn related outcomes with shared predictors
>* Cross validate regularization for better generalization

>* Balance shrinkage to avoid underfitting or noise
>* Choose settings that predict all tasks well

>* Check accuracy and shared selected features
>* Validate shared structure with domain knowledge



## **3. Optimization Diagnostics**

### **3.1. Coordinate Descent**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_03_01.jpg?v=1783959372" width="250">



>* Updates one coefficient at a time
>* Shrinks unimportant features to zero

>* Standardize features for fair coordinate updates
>* Correlations shape Lasso and Elastic Net selection

>* Convergence warnings require careful coefficient checks
>* Unstable sparsity may signal preprocessing issues



### **3.2. Convergence Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_03_02.jpg?v=1783959374" width="250">



>* Check optimization has settled reliably
>* Avoid interpreting unstable coefficients as meaningful

>* Treat warnings as optimization diagnostic signals
>* Standardize, tune, and check collinearity

>* Trust coefficients only after solid convergence
>* Check sparsity stability before selecting features



### **3.3. Coefficient Paths**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_B/image_03_03.jpg?v=1783959376" width="250">



>* Track coefficients as regularization strength changes
>* Early stable features signal stronger predictors

>* Track entry order and coefficient stability
>* Use paths to diagnose correlated predictors

>* Show sparse feature choices from regularization
>* Reveal scaling, convergence, and instability issues



# <font color="#418FDE" size="6.5" uppercase>**Lasso Elastic Net**</font>


In this lecture, you learned to:
- Fit Lasso and Elastic Net models and interpret sparse coefficients. 
- Use cross-validation to select alpha and l1_ratio values. 
- Analyze convergence warnings, coefficient paths, and sparsity-based feature selection. 

In the next Module (Module 10), we will go over 'Advanced Linear'